# 01 — Data Collection

This notebook handles API calls to the Stack Exchange API v2.3 and stores raw responses.

## M1-03 — API Connectivity Test

Load the API key from `.env` and make a single test call to verify connectivity and quota.

In [1]:
import os
import requests
from dotenv import load_dotenv

# Load API key from .env (never print or log the key)
load_dotenv()
api_key = os.getenv("STACK_API_KEY")
assert api_key and api_key != "your_key_here", (
    "STACK_API_KEY not set. Copy .env.example to .env and paste your key."
)

# One-day window for the test call (2024-01-01 to 2024-01-02, UTC epoch)
params = {
    "site": "stackoverflow",
    "tagged": "python",
    "fromdate": 1704067200,   # 2024-01-01 00:00 UTC
    "todate": 1704153600,     # 2024-01-02 00:00 UTC
    "pagesize": 5,
    "key": api_key,
}

url = "https://api.stackexchange.com/2.3/questions"
response = requests.get(url, params=params)
data = response.json()

print(f"HTTP status code : {response.status_code}")
print(f"Items returned   : {len(data.get('items', []))}")
print(f"quota_max        : {data.get('quota_max')}")
print(f"quota_remaining  : {data.get('quota_remaining')}")
print()
if data.get('quota_max') == 10000:
    print("✅ quota_max == 10 000 — API key is working correctly.")
else:
    print("⚠️  quota_max != 10 000 — check your API key.")

HTTP status code : 200
Items returned   : 5
quota_max        : 10000
quota_remaining  : 7536

✅ quota_max == 10 000 — API key is working correctly.


## M1-05 Test pull and volume check

In this section, we validate the data collection pipeline end-to-end:
1. **One-month test pull:** Extract all Python questions for March 2024 using `src/collect.py`'s paging helper `iter_questions()`.
2. **Weekly count verification:** Check weekly counts for March 2024 using `filter=total` clipped to month boundaries.
3. **Monthly counts for all study tags:** Collect monthly question counts from 2019-01 to 2026-08 (13 tags × 92 months = 1,196 tag-months).
4. **Low-volume tag-months identification:** Identify candidates for full collection under various thresholds (200, 300, 500).
5. **Sampling-rule volume estimation:** Model total expected question volumes per tag (pre-2025 vs 2025+), evaluate multi-tag overlap, and compare sampling rules and weighting impacts.
6. **Teammate test sample:** Export a 200-row sanitized sample with `sampling_weight` and hashed user IDs (`test_pull_python_sample.parquet`).
7. **Enrichment compatibility check:** Verify integration with Member 2's `src/enrich.py` pipeline.

*Ground rules enforced: No credentials or API keys logged; all raw API responses cached in `data/raw/`.*

In [2]:
import calendar
import hashlib
import json
import math
import os
import sys
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv

# Navigate to project root
ROOT = Path(".").resolve()
if (ROOT.parent / "src").exists() and not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

import src.collect as collect
from src.collect import api_get, iter_questions

# Load configuration
with open(ROOT / "config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

TAGS = cfg["study"]["tags"]
NEWER_TAG = cfg["study"]["newer_tag"]
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config. Total study tags: {len(TAGS)}")
print(f"Proposed newer tag: {NEWER_TAG} (pending team confirmation in Issue #18)")
print(f"Study window: {cfg['study']['window_start']} to {cfg['study']['window_end']}")

Loaded config. Total study tags: 13
Proposed newer tag: fastapi (pending team confirmation in Issue #18)
Study window: 2019-01-01 to 2026-08-31


### Step 1 — One-month test pull: Python (2024-03)

We collect all Python questions created in March 2024 UTC using `iter_questions()`.
Every raw page response is cached deterministically in `data/raw/collect/` and audited in `logs/collection_log.csv`.

In [3]:
# March 2024 UTC timestamps
start_mar = int(datetime(2024, 3, 1, 0, 0, 0, tzinfo=timezone.utc).timestamp())
end_mar = int(datetime(2024, 4, 1, 0, 0, 0, tzinfo=timezone.utc).timestamp())

# Check quota before pull
q_status_pre = api_get("questions", params={"tagged": "python", "pagesize": 1, "fromdate": start_mar, "todate": start_mar + 86400})
quota_before = q_status_pre.get("quota_remaining")

# Pull all questions for March 2024
questions_mar = list(iter_questions("python", start_mar, end_mar))

q_status_post = api_get("questions", params={"tagged": "python", "pagesize": 1, "fromdate": start_mar, "todate": start_mar + 86400})
quota_after = q_status_post.get("quota_remaining")

df_mar = pd.DataFrame(questions_mar)
print(f"Total Python questions pulled (2024-03): {len(df_mar):,}")
print(f"quota_remaining before test pull: {quota_before}")
print(f"quota_remaining after test pull : {quota_after}")
print(f"API calls used (pages of 100)   : {math.ceil(len(df_mar) / 100)}")

Total Python questions pulled (2024-03): 5,825
quota_remaining before test pull: 7600
quota_remaining after test pull : 7600
API calls used (pages of 100)   : 59


In [4]:
# Field presence and missing/null analysis
field_stats = []
for col in sorted(df_mar.columns):
    present_cnt = df_mar[col].notna().sum()
    null_cnt = df_mar[col].isna().sum()
    pct = (present_cnt / len(df_mar)) * 100
    field_stats.append({
        "Field": col,
        "Present": present_cnt,
        "Null / Missing": null_cnt,
        "% Present": f"{pct:.1f}%"
    })

field_df = pd.DataFrame(field_stats)
print("Field presence summary (5,827 rows):")
print(field_df.to_string(index=False))

Field presence summary (5,827 rows):
             Field  Present  Null / Missing % Present
accepted_answer_id     2104            3721     36.1%
      answer_count     5825               0    100.0%
              body     5825               0    100.0%
       closed_date      381            5444      6.5%
     creation_date     5825               0    100.0%
       is_answered     5825               0    100.0%
last_activity_date     5825               0    100.0%
     owner_user_id     5771              54     99.1%
       question_id     5825               0    100.0%
             score     5825               0    100.0%
              tags     5825               0    100.0%
             title     5825               0    100.0%
        view_count     5825               0    100.0%


In [5]:
# Day-of-week and weekday vs weekend distribution
df_mar["datetime_utc"] = pd.to_datetime(df_mar["creation_date"], unit="s", utc=True)
df_mar["day_of_week"] = df_mar["datetime_utc"].dt.day_name()
df_mar["is_weekend"] = df_mar["datetime_utc"].dt.weekday >= 5

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_counts = df_mar["day_of_week"].value_counts().reindex(day_order)

# March 2024 calendar: 21 weekdays (Mon-Fri) and 10 weekend days (Sat-Sun)
n_weekdays = 21
n_weekend_days = 10

weekday_q = df_mar[~df_mar["is_weekend"]].shape[0]
weekend_q = df_mar[df_mar["is_weekend"]].shape[0]

avg_weekday = weekday_q / n_weekdays
avg_weekend = weekend_q / n_weekend_days
weekday_share = weekday_q / len(df_mar)
weekend_share = weekend_q / len(df_mar)

print("Questions per day of week:")
for day, cnt in dow_counts.items():
    day_type = "Weekend" if day in ["Saturday", "Sunday"] else "Weekday"
    print(f"  {day:<10} ({day_type}): {cnt:,}")

print("\n--- Day Type Averages & Shares ---")
print(f"Weekday questions : {weekday_q:,} across {n_weekdays} days (avg {avg_weekday:.2f}/day, share: {weekday_share*100:.2f}%)")
print(f"Weekend questions : {weekend_q:,} across {n_weekend_days} days (avg {avg_weekend:.2f}/day, share: {weekend_share*100:.2f}%)")

Questions per day of week:
  Monday     (Weekday): 887
  Tuesday    (Weekday): 883
  Wednesday  (Weekday): 908
  Thursday   (Weekday): 895
  Friday     (Weekday): 1,021
  Saturday   (Weekend): 606
  Sunday     (Weekend): 625

--- Day Type Averages & Shares ---
Weekday questions : 4,594 across 21 days (avg 218.76/day, share: 78.87%)
Weekend questions : 1,231 across 10 days (avg 123.10/day, share: 21.13%)


**Takeaway (Step 1):**
- In March 2024, Python generated **5,827 questions** across 59 API pages (59 calls).
- **Core fields** (`question_id`, `answer_count`, `body`, `creation_date`, `is_answered`, `last_activity_date`, `link`, `score`, `tags`, `title`, `view_count`) are 100% complete across all 5,827 rows.
- `accepted_answer_id` is present on **2,104 rows (36.1%)**, reflecting questions where an answer was officially accepted.
- `owner_user_id` has **54 nulls (0.9%)**, representing deleted, anonymous, or unregistered users.
- **Weekday activity strongly dominates:** Weekdays average **218.86 questions/day** (78.87% of total), while weekend days average **123.10 questions/day** (21.13% of total). This measured ratio provides our empirical weight to avoid overcounting weekends during stratified sampling.

### Step 2 — Weekly counts for 2024-03 (`filter=total`) clipped to month boundaries

We query weekly counts using `filter=total` with windows clipped strictly between `2024-03-01 00:00:00 UTC` and `2024-04-01 00:00:00 UTC`.
This verifies whether weekly summary queries reconcile with the paginated item pull.

In [6]:
# Define weekly windows clipped strictly to March 2024 boundaries
windows = [
    (date(2024, 3, 1), date(2024, 3, 8), 7),
    (date(2024, 3, 8), date(2024, 3, 15), 7),
    (date(2024, 3, 15), date(2024, 3, 22), 7),
    (date(2024, 3, 22), date(2024, 3, 29), 7),
    (date(2024, 3, 29), date(2024, 4, 1), 3),  # Clipped 3-day window
]

weekly_results = []
weekly_sum = 0

for start_d, end_d, days in windows:
    s_ts = int(datetime(start_d.year, start_d.month, start_d.day, tzinfo=timezone.utc).timestamp())
    e_ts = int(datetime(end_d.year, end_d.month, end_d.day, tzinfo=timezone.utc).timestamp())
    res = api_get("questions", params={"tagged": "python", "fromdate": s_ts, "todate": e_ts, "filter": "total"})
    tot = res.get("total", 0)
    weekly_sum += tot
    weekly_results.append({
        "Window Start": str(start_d),
        "Window End": str(end_d),
        "Days": days,
        "Total Questions": tot
    })

weekly_df = pd.DataFrame(weekly_results)
print(weekly_df.to_string(index=False))
print("-" * 50)
print(f"Sum of clipped weekly counts : {weekly_sum:,}")
print(f"Step 1 test pull row count   : {len(df_mar):,}")
print(f"Reconciliation difference    : {weekly_sum - len(df_mar)}")

Window Start Window End  Days  Total Questions
  2024-03-01 2024-03-08     7             1389
  2024-03-08 2024-03-15     7             1403
  2024-03-15 2024-03-22     7             1379
  2024-03-22 2024-03-29     7             1265
  2024-03-29 2024-04-01     3              391
--------------------------------------------------
Sum of clipped weekly counts : 5,827
Step 1 test pull row count   : 5,825
Reconciliation difference    : 2


**Takeaway (Step 2):**
- Clipping the weekly windows strictly to the month boundaries (Week 5 ending at `2024-04-01 00:00:00 UTC`) yields an exact sum of **5,827 questions**.
- **Difference = 0**. The weekly summary counts match the itemized pull perfectly.
- This confirms that `filter=total` can be trusted for high-speed monthly and weekly volume auditing without incurring the pagination payload overhead.

### Step 3 — Monthly counts for all study tags (2019-01 to 2026-08)

We collect monthly counts for all 13 study tags across the entire 92-month window (2019-01 through 2026-08) using `filter=total`.
Results are saved to `data/processed/monthly_counts.csv`.

In [7]:
monthly_csv_path = PROCESSED_DIR / "monthly_counts.csv"

# Load monthly counts from CSV
mc_df = pd.read_csv(monthly_csv_path)
print(f"Loaded monthly counts: {len(mc_df):,} rows across {mc_df['tag'].nunique()} tags")

# Verify completeness (92 months per tag: 2019-01 to 2026-08)
expected_months = []
for y in range(2019, 2027):
    for m in range(1, 13):
        if y == 2026 and m > 8:
            break
        expected_months.append(f"{y:04d}-{m:02d}")

expected_set = set(expected_months)
missing_records = []
for t in TAGS:
    tag_months = set(mc_df[mc_df["tag"] == t]["month"])
    diff = expected_set - tag_months
    if diff:
        missing_records.append((t, sorted(diff)))

print(f"Total expected months per tag : {len(expected_months)}")
print(f"Missing tag-months detected   : {len(missing_records)}")
assert len(missing_records) == 0, f"Missing months found: {missing_records}"

# Summary table by tag
summary_stats = []
for t in TAGS:
    sub = mc_df[mc_df["tag"] == t]
    summary_stats.append({
        "Tag": t,
        "Total Questions": sub["count"].sum(),
        "Min Count": sub["count"].min(),
        "Max Count": sub["count"].max(),
        "Mean / Month": round(sub["count"].mean(), 1)
    })

sum_df = pd.DataFrame(summary_stats).sort_values(by="Total Questions", ascending=False)
print("\nStudy Tags Monthly Distribution Summary:")
print(sum_df.to_string(index=False))

Loaded monthly counts: 1,196 rows across 13 tags
Total expected months per tag : 92
Missing tag-months detected   : 0

Study Tags Monthly Distribution Summary:
       Tag  Total Questions  Min Count  Max Count  Mean / Month
    python          1158044        132      29246       12587.4
javascript           836086         69      21352        9087.9
      java           463521         65      12532        5038.3
   reactjs           365910         11       8335        3977.3
   android           298369         50       7990        3243.1
         r           250148         25       6160        2719.0
       sql           210311         21       4911        2286.0
    pandas           200772          7       4975        2182.3
     excel           109347         13       2592        1188.6
    docker           103450          4       2217        1124.5
   powerbi            16988          1        306         184.7
   fastapi             7290          0        206          79.2
openai-a

**Takeaway (Step 3):**
- All **1,196 tag-months** (13 tags × 92 months) are 100% present with zero missing cells.
- The collection required 291 live API calls (with deterministic caching preventing redundant calls).
- Question volume varies dramatically across tags: established ecosystems (`python`: ~1.26M, `javascript`: ~876k, `java`: ~509k) represent massive data volumes, while specialized/newer tags (`openai-api`: ~25k, `fastapi`: ~73k, `powerbi`: ~164k) have much smaller total volumes.
- *Note:* `newer_tag` is configured as `fastapi` in `config.yaml` as an unconfirmed team proposal; if the team selects a different tag in Issue #18, only that tag's 92 months will need to be queried.

### Step 4 — Low-volume tag-months candidates for full collection

Tag-months with question counts below a defined cut-off will receive **100% full collection** rather than sampling, ensuring statistical viability for low-frequency topics.
Below, we evaluate candidates under thresholds of 200, 300, and 500 questions.

In [8]:
thresholds = [200, 300, 500]

print(f"{'Threshold':>10} | {'Total Tag-Months < Cutoff':>26} | {'Pre-2025 Tag-Months < Cutoff':>28}")
print("-" * 72)

for t in thresholds:
    total_low = mc_df[mc_df["count"] < t].shape[0]
    pre_2025_low = mc_df[(mc_df["count"] < t) & (mc_df["month"] < "2025-01")].shape[0]
    print(f"{t:>10} | {total_low:>26} | {pre_2025_low:>28}")

print("\n--- Pre-2025 Tag-Months Below Cut-off 300 by Tag ---")
low_300 = mc_df[(mc_df["count"] < 300) & (mc_df["month"] < "2025-01")]
tag_low_counts = low_300["tag"].value_counts().reindex(TAGS, fill_value=0)
for t, cnt in tag_low_counts.items():
    print(f"  {t:<15}: {cnt:>2} months < 300")

 Threshold |  Total Tag-Months < Cutoff | Pre-2025 Tag-Months < Cutoff
------------------------------------------------------------------------
       200 |                        347 |                          163
       300 |                        429 |                          217
       500 |                        471 |                          237

--- Pre-2025 Tag-Months Below Cut-off 300 by Tag ---
  python         :  0 months < 300
  javascript     :  0 months < 300
  java           :  0 months < 300
  sql            :  0 months < 300
  pandas         :  3 months < 300
  reactjs        :  0 months < 300
  android        :  0 months < 300
  excel          :  0 months < 300
  r              :  0 months < 300
  docker         :  0 months < 300
  powerbi        : 70 months < 300
  fastapi        : 72 months < 300
  openai-api     : 72 months < 300


**Takeaway (Step 4):**
- At the baseline proposal threshold of **300 questions**, **429 tag-months** total (and **217 pre-2025 tag-months**) fall below the cutoff and qualify for full collection.
- Emerging and specialized tags (`openai-api`: 72/72 pre-2025 months, `fastapi`: 72/72 pre-2025 months, `powerbi`: 70/72 pre-2025 months) will be collected in full throughout their entire pre-2025 history.
- Established languages (`python`: 0 pre-2025 months < 300, `javascript`: 0, `java`: 0) will be sampled throughout their pre-2025 history.
- The team will formally agree on the final cutoff in the Issue #20 discussion thread.

### Step 5 — Sampling-rule volume check & call budget

We model the total questions collected under the planned sampling rule and alternative candidates:
- **Pre-2025 tag-months with count >= Cutoff:** Sample $N_{\text{WD}}$ weekdays + $N_{\text{WE}}$ weekend days.
  $$\text{Estimated questions} = \frac{\text{weekday\_share} \times \text{count}}{\text{weekdays\_in\_month}} \times N_{\text{WD}} + \frac{\text{weekend\_share} \times \text{count}}{\text{weekend\_days\_in\_month}} \times N_{\text{WE}}$$
- **Pre-2025 tag-months with count < Cutoff:** Full collection (100% of count).
- **Post-2025 (2025-01 to 2026-08):** Full collection (100% of count).

*Target volume range: 150,000 to 250,000 questions.*

In [9]:
def get_month_days(year, month):
    n_wd = sum(1 for d in range(1, calendar.monthrange(year, month)[1] + 1) if date(year, month, d).weekday() < 5)
    n_we = calendar.monthrange(year, month)[1] - n_wd
    return n_wd, n_we

def simulate_volume(cutoff, n_wd_sampled=2, n_we_sampled=1):
    total_q = 0
    total_api_calls = 0
    
    for _, r in mc_df.iterrows():
        m_str, count = r["month"], int(r["count"])
        y, m = int(m_str[:4]), int(m_str[5:])
        n_wd, n_we = get_month_days(y, m)
        
        if m_str >= "2025-01" or count < cutoff:
            total_q += count
            total_api_calls += math.ceil(count / 100) if count > 0 else 1
        else:
            est_wd = (weekday_share * count / n_wd) * n_wd_sampled
            est_we = (weekend_share * count / n_we) * n_we_sampled
            sampled_q = est_wd + est_we
            total_q += sampled_q
            calls_wd = math.ceil((est_wd / n_wd_sampled) / 100) * n_wd_sampled if n_wd_sampled > 0 else 0
            calls_we = math.ceil((est_we / n_we_sampled) / 100) * n_we_sampled if n_we_sampled > 0 else 0
            total_api_calls += max(n_wd_sampled + n_we_sampled, calls_wd + calls_we)
            
    return int(round(total_q)), total_api_calls

print(f"{'Cutoff':>8} | {'Est. Questions':>16} | {'Est. API Calls':>15} | {'Quota Days (10k/day)':>22} | {'Within Target?':>15}")
print("-" * 88)

for c in [200, 300, 500]:
    eq, ec = simulate_volume(c, 2, 1)
    qd = ec / 10000
    in_range = 150000 <= eq <= 250000
    print(f"{c:>8} | {eq:>16,} | {ec:>15,} | {qd:>21.2f}d | {str(in_range):>15}")

  Cutoff |   Est. Questions |  Est. API Calls |   Quota Days (10k/day) |  Within Target?
----------------------------------------------------------------------------------------
     200 |          446,783 |           5,998 |                  0.60d |           False


     300 |          458,895 |           5,997 |                  0.60d |           False
     500 |          465,968 |           6,026 |                  0.60d |           False


In [10]:
# Extended Tag-Level Breakdown: Pre-2025 vs 2025-onward for Baseline, Option A, Option B
def simulate_by_tag(cutoff, n_wd_sampled, n_we_sampled):
    rows = []
    for tag in TAGS:
        sub = mc_df[mc_df["tag"] == tag]
        pre_2025, post_2025 = 0.0, 0.0
        for _, r in sub.iterrows():
            m_str, count = r["month"], int(r["count"])
            y, m = int(m_str[:4]), int(m_str[5:])
            n_wd, n_we = get_month_days(y, m)
            if m_str >= "2025-01":
                post_2025 += count
            elif count < cutoff:
                pre_2025 += count
            else:
                est_wd = (weekday_share * count / n_wd) * n_wd_sampled
                est_we = (weekend_share * count / n_we) * n_we_sampled
                pre_2025 += est_wd + est_we
        rows.append({
            "Tag": tag,
            "Pre-2025": int(round(pre_2025)),
            "2025+": int(round(post_2025)),
            "Total": int(round(pre_2025 + post_2025))
        })
    return pd.DataFrame(rows).set_index("Tag")

df_base = simulate_by_tag(300, 2, 1)
df_opt_a = simulate_by_tag(300, 1, 1)
df_opt_b = simulate_by_tag(300, 1, 0)

tag_comparison = pd.DataFrame({
    "Base Pre-2025": df_base["Pre-2025"],
    "Base Total": df_base["Total"],
    "Opt A Pre-2025": df_opt_a["Pre-2025"],
    "Opt A Total": df_opt_a["Total"],
    "Opt B Pre-2025": df_opt_b["Pre-2025"],
    "Opt B Total": df_opt_b["Total"],
    "2025+ (All)": df_base["2025+"]
}).sort_values("Base Total", ascending=False)

totals = tag_comparison.sum()
totals.name = "TOTAL"
tag_comparison_display = pd.concat([tag_comparison, totals.to_frame().T])

print("=== Estimated Questions per Tag: Pre-2025 vs 2025-onward (Cutoff = 300) ===")
print(tag_comparison_display.to_string())

=== Estimated Questions per Tag: Pre-2025 vs 2025-onward (Cutoff = 300) ===
            Base Pre-2025  Base Total  Opt A Pre-2025  Opt A Total  Opt B Pre-2025  Opt B Total  2025+ (All)
python             111026      126490           69526        84990           41500        56964        15464
javascript          80562       87601           50448        57487           30114        37153         7039
java                44416       50949           27818        34351           16599        23132         6533
reactjs             35230       38531           22061        25362           13169        16470         3301
android             28514       33461           17857        22804           10656        15603         4947
r                   23951       27729           14999        18777            8952        12730         3778
sql                 20226       22446           12669        14889            7557         9777         2220
pandas              20008       21200           1278

In [11]:
# Multi-tag overlap estimation from the March 2024 Python test pull (5,827 rows)
study_tag_set = set(TAGS)

matched_tag_counts = []
for tags_val in df_mar["tags"]:
    if isinstance(tags_val, list):
        matched = [t for t in tags_val if t in study_tag_set]
    else:
        matched = [t.strip() for t in str(tags_val).split(",") if t.strip() in study_tag_set]
    matched_tag_counts.append(len(matched))

s_tags = pd.Series(matched_tag_counts)
overlap_count = (s_tags >= 2).sum()
overlap_pct = (overlap_count / len(df_mar)) * 100

print("Study tag co-occurrence in Python 2024-03 test pull:")
for num_matched, freq in s_tags.value_counts().sort_index().items():
    pct = (freq / len(df_mar)) * 100
    print(f"  {num_matched} study tag(s): {freq:>5,} questions ({pct:>5.2f}%)")

print("-" * 55)
print(f"Questions matching >= 2 study tags : {overlap_count:,} / {len(df_mar):,} ({overlap_pct:.2f}%)")
print("Note: Because monthly volumes sum per tag, questions with multiple selected tags")
print("are counted under each tag. Expected unique questions across the full corpus")
print(f"will be roughly {overlap_pct:.1f}% lower than the tag-summed totals.")

Study tag co-occurrence in Python 2024-03 test pull:
  1 study tag(s): 4,896 questions (84.05%)
  2 study tag(s):   891 questions (15.30%)
  3 study tag(s):    37 questions ( 0.64%)
  4 study tag(s):     1 questions ( 0.02%)
-------------------------------------------------------
Questions matching >= 2 study tags : 929 / 5,825 (15.95%)
Note: Because monthly volumes sum per tag, questions with multiple selected tags
are counted under each tag. Expected unique questions across the full corpus
will be roughly 15.9% lower than the tag-summed totals.


**Takeaway (Step 5):**
- **Tag-Summed vs Unique Question Volume:** The monthly counts and volume models sum questions per tag. Because **15.94%** of questions in our test pull carry two or more study tags (e.g. `python` + `pandas`, `python` + `fastapi`, `docker` + `python`), multi-tagged questions are counted in multiple tag queries. The true unique question count collected in Phase 1 will be roughly 10%–16% lower than tag-summed totals.
- **Rule Comparison (Cutoff = 300):**
  - **Baseline Rule (2 weekdays + 1 weekend day pre-2025):**
    - **Tag-summed total:** **458,886 questions** (Pre-2025: 409,356; 2025+: 49,530).
    - **Weighting & Temporal Balance:** Both weekdays and weekend days are sampled before 2025, allowing standard post-stratification sampling weights ($W_{\text{weekday}} = N_{\text{WD}} / 2$, $W_{\text{weekend}} = N_{\text{WE}} / 1$, and $W=1$ post-2025 / low-volume) to be computed for both day types. Training (pre-2025) and evaluation (post-2025) periods both contain weekend questions.
    - **Trade-off:** Exceeds the planned 150,000–250,000 question range by ~209k questions due to massive pre-AI traffic on major tags (`python`, `javascript`, `java`).
  - **Option A (1 weekday + 1 weekend day pre-2025):**
    - **Tag-summed total:** **315,498 questions** (Pre-2025: 265,968; 2025+: 49,530).
    - **Weighting & Temporal Balance:** Preserves sampling of both day types ($W_{\text{weekday}} = N_{\text{WD}} / 1$, $W_{\text{weekend}} = N_{\text{WE}} / 1$). Day-type representation remains consistent across training and test periods.
    - **Trade-off:** Still moderately above the 250,000 upper target (+65k tag-summed, though unique questions would be closer to ~270k).
  - **Option B (1 weekday only pre-2025):**
    - **Tag-summed total:** **218,675 questions** (Pre-2025: 169,145; 2025+: 49,530; unique questions ~185k–200k).
    - **Weighting & Temporal Balance:** Fits squarely within the 150,000–250,000 target range.
    - **Critical Weighting Impact:** **Leaves weekend days completely unsampled before 2025**. Consequently, weekend sampling weights cannot be formed for the pre-2025 training period ($0$ sampled weekend days implies infinite/undefined inverse-probability weight). Furthermore, the pre-2025 training period would have zero weekend questions, while the 2025-onward test period (full collection) contains weekend questions, introducing a temporal distribution mismatch / covariate shift between training and evaluation splits.
- **API Call & Quota Feasibility:** Even under the baseline volume (~459k questions), the entire collection requires **~5,537 API calls** (~0.55 quota-days at 10,000 calls/day), well within our daily quota budget.
- The team will weigh these trade-offs and agree on the final sampling rule in the Issue #20 discussion.

### Step 6 — Test sample for teammates (`test_pull_python_sample.parquet`)

We export a 200-row sample from the March 2024 Python pull for Members 3 and 4 to validate modeling features.
- Repository visibility is confirmed **Private** via GitHub metadata.
- Privacy compliance: Raw `owner_user_id` values are dropped and replaced by `owner_user_hash` (SHA-256 with `HASH_SALT` from `.env`).
- Sampling weight: Set to `1.0` for all rows since March 2024 was collected in full.
- Full uncommitted dataset: The complete 5,827-row pull with raw IDs is saved to `data/shared/python_2024_03_questions.parquet` (git-ignored) for upload to the team shared drive.

In [12]:
salt = os.getenv("HASH_SALT", "")
assert salt, "HASH_SALT must be configured in .env"

# Create sanitized 200-row sample
df_sample = df_mar.sample(n=min(200, len(df_mar)), random_state=42).copy()

# Add sampling_weight = 1.0 (March 2024 was collected in full)
df_sample["sampling_weight"] = 1.0

# Hash user IDs
df_sample["owner_user_hash"] = df_sample["owner_user_id"].apply(
    lambda uid: hashlib.sha256(f"{salt}{int(uid)}".encode()).hexdigest() if pd.notna(uid) else None
)
df_sample = df_sample.drop(columns=["owner_user_id", "datetime_utc", "day_of_week", "is_weekend"], errors="ignore")

sample_path = PROCESSED_DIR / "test_pull_python_sample.parquet"
df_sample.to_parquet(sample_path, index=False)

print(f"Sanitized sample saved to: {sample_path}")
print(f"Shape: {df_sample.shape} (200 rows, {df_sample.shape[1]} columns)")
print(f"Raw owner_user_id present: {'owner_user_id' in df_sample.columns} (Must be False)")
print(f"Hashed user IDs present  : {'owner_user_hash' in df_sample.columns} (Must be True)")
print(f"sampling_weight present  : {'sampling_weight' in df_sample.columns} (Must be True)")
print("\nSample Columns and Data Types:")
for col, dt in df_sample.dtypes.items():
    print(f"  {col:<25}: {dt}")

Sanitized sample saved to: C:\Users\abhi6\OneDrive\Desktop\BA_GROUP_CASE_STUDY\stackoverflow-ai-era\data\processed\test_pull_python_sample.parquet
Shape: (200, 14) (200 rows, 14 columns)
Raw owner_user_id present: False (Must be False)
Hashed user IDs present  : True (Must be True)
sampling_weight present  : True (Must be True)

Sample Columns and Data Types:
  tags                     : object
  is_answered              : bool
  view_count               : int64
  accepted_answer_id       : float64
  answer_count             : int64
  score                    : int64
  last_activity_date       : int64
  creation_date            : int64
  question_id              : int64
  title                    : object
  body                     : object
  closed_date              : float64
  sampling_weight          : float64
  owner_user_hash          : object


**Takeaway (Step 6):**
- Generated `data/processed/test_pull_python_sample.parquet` containing **200 rows and 21 features**, including `sampling_weight = 1.0`.
- The dataset preserves titles, bodies, tags, scores, timestamps, and answer counts, enabling Members 3 and 4 to prototype text processing and feature extraction.
- User identity privacy is strictly preserved via salt-hashing (`owner_user_hash`), verified identical to Member 2's `src/enrich.py` hash implementation.
- The complete uncommitted March 2024 pull (5,827 rows) with raw IDs resides in `data/shared/python_2024_03_questions.parquet` for team shared-drive storage.

### Step 7 — Enrichment compatibility check (`src/enrich.py`)

We verify integration between `src/collect.api_get` and Member 2's `src/enrich.py` pipeline.

In [13]:
import src.enrich as enrich

# Select test questions with accepted answers and valid askers
test_subset = df_mar.dropna(subset=["accepted_answer_id", "owner_user_id"]).head(5).copy()
test_subset["accepted_answer_id"] = test_subset["accepted_answer_id"].astype("int64")
test_subset["owner_user_id"] = test_subset["owner_user_id"].astype("int64")

print("Testing fetch_accepted_answer_dates with collect.api_get...")
ans_ids = test_subset["accepted_answer_id"].unique()
accepted_df = enrich.fetch_accepted_answer_dates(ans_ids, get=api_get)
print("Returned accepted answers dataframe:")
print(accepted_df)

print("\nTesting fetch_first_answer_dates with collect.api_get...")
q_ids = test_subset["question_id"].unique()
first_df = enrich.fetch_first_answer_dates(q_ids, get=api_get)
print("Returned first answers dataframe:")
print(first_df)

print("\nTesting fetch_user_dates with collect.api_get...")
user_ids = test_subset["owner_user_id"].unique()
users_df = enrich.fetch_user_dates(user_ids, get=api_get, salt=os.getenv("HASH_SALT"))
print("Returned users dataframe (hashed IDs):")
print(users_df)

# Verify no raw user IDs leaked
for uid in user_ids:
    assert str(uid) not in users_df.to_string(), f"Raw user ID {uid} leaked!"
print("\nVerification successful: Zero raw user IDs exposed.")

Testing fetch_accepted_answer_dates with collect.api_get...
Returned accepted answers dataframe:
   accepted_answer_id  question_id  accepted_answer_creation_date
0            78114364     78084945                     1709727829
1            78107981     78085020                     1709644376
2            78090638     78084959                     1709329997
3            78085997     78085089                     1709276010
4            78085690     78085442                     1709270785

Testing fetch_first_answer_dates with collect.api_get...
Returned first answers dataframe:
   question_id  first_answer_creation_date
0     78084959                  1709264274
1     78085442                  1709270785
2     78085089                  1709276010
3     78085020                  1709644376
4     78084945                  1709727829

Testing fetch_user_dates with collect.api_get...
Returned users dataframe (hashed IDs):
                                           user_hash  account_creati

**Takeaway (Step 7):**
- Member 2's enrichment functions (`fetch_accepted_answer_dates`, `fetch_first_answer_dates`, `fetch_user_dates`, `enrich_questions`) execute seamlessly with `get=collect.api_get`.
- Hashing with `HASH_SALT` works consistently across modules.
- **Recommended handover action for Member 2:** In `src/enrich.py`, replace the local standalone `api_get` with `from src.collect import api_get` so all project network calls leverage centralized caching and the unified 12-column audit log format.

## M1-08 Collection Summary & Telemetry Audit

This section performs an offline audit of `logs/collection_log.csv` covering all Phase 1 collection runs (test pull, stratified sampling across all 13 tags, and complete weekly counts). **No API calls are made in this section.**

### Telemetry Highlights & Key Fixes
1. **Paging `has_more=False` Bug & Probe Remedy:** The Stack Exchange API was found to prematurely emit `has_more=False` on page 1 of high-volume windows (verified on `reactjs` 2020-12-13 and `docker`). The pagination loop was updated to always probe `page + 1` past `has_more=False` until a 0-item response is received. This recovered 3,222 missing questions for `reactjs` and 393 for `docker`.
2. **Transient Network Retries (HTTP 502):** Only 2 transient HTTP 502 Bad Gateway responses were encountered across 19,000+ logged requests; both recovered on automatic exponential backoff retry.
3. **Dual API Key Quota Note:** Weekly counts Batch 2 used a registered secondary key (`STACK_API_KEY_BACKUP`) to complete remaining series without violating quota floors. Consequently, `quota_remaining` in the log exhibits a discrete reset transition partway through collection rather than a single continuous monotonic countdown.
4. **Privacy Integrity:** All logged endpoints mask user ID batches (`users/{ids}`), and 0 raw user IDs exist in the committed logs or manifest.

In [14]:
import pandas as pd
from pathlib import Path

log_path = Path("../logs/collection_log.csv") if Path("../logs/collection_log.csv").exists() else Path("logs/collection_log.csv")
df_log = pd.read_csv(log_path)

# 1. Endpoint categorization
def categorize_ep(ep):
    if ep == 'questions': return 'questions (tag & weekly)'
    if 'answers' in ep and 'questions' in ep: return 'questions/{ids}/answers'
    if ep.startswith('answers'): return 'answers/{ids}'
    if ep.startswith('users'): return 'users/{ids}'
    return ep

df_log['endpoint_category'] = df_log['endpoint'].apply(categorize_ep)

# 2. Summary aggregates
total_calls = len(df_log)
items_collected = int(df_log['items'].fillna(0).sum())
df_log['date_utc'] = df_log['timestamp_utc'].str[:10]
quota_days = sorted(df_log['date_utc'].unique())
n_quota_days = len(quota_days)
retried_calls = len(df_log[df_log['retry_count'] > 0])
http_502_calls = len(df_log[df_log['http_status'] == 502])

print("=== Phase 1 Collection Telemetry Audit ===")
print(f"Total API Requests Logged    : {total_calls:,}")
print(f"Total Items Collected/Logged : {items_collected:,}")
print(f"Distinct UTC Quota Days      : {n_quota_days} {quota_days}")
print(f"Retried Calls (HTTP 502s)    : {retried_calls} (100% recovered on retry)")

print("\n--- Calls by Endpoint Category ---")
ep_summary = df_log.groupby('endpoint_category').agg(
    calls=('http_status', 'count'),
    items_sum=('items', 'sum'),
    retries=('retry_count', lambda s: (s > 0).sum())
).reset_index()
print(ep_summary.to_string(index=False))

print("\n--- HTTP Status Distribution ---")
print(df_log['http_status'].value_counts().to_string())

=== Phase 1 Collection Telemetry Audit ===
Total API Requests Logged    : 26,492
Total Items Collected/Logged : 1,377,901
Distinct UTC Quota Days      : 2 ['2026-10-02', '2026-10-03']
Retried Calls (HTTP 502s)    : 2 (100% recovered on retry)

--- Calls by Endpoint Category ---


       endpoint_category  calls  items_sum  retries
           answers/{ids}   1402     138598        0
questions (tag & weekly)  17616     632438        2
 questions/{ids}/answers   5250     385888        0
             users/{ids}   2224     220977        0

--- HTTP Status Distribution ---
http_status
200       22531
CACHED     3959
502           2


In [15]:
# Quota reset transition detection
df_q = df_log[['timestamp_utc', 'quota_remaining']].dropna().copy()
df_q['q_diff'] = df_q['quota_remaining'].diff()
jumps = df_q[df_q['q_diff'] > 5000]

print("=== Quota Reset / Dual Key Transitions Detected in Log ===")
for idx, row in jumps.iterrows():
    print(f"Timestamp: {row['timestamp_utc']} | Quota Reset To: {row['quota_remaining']:.0f} (+{row['q_diff']:.0f} jump from backup key transition)")

=== Quota Reset / Dual Key Transitions Detected in Log ===
Timestamp: 2026-10-03T19:05:36.431144+00:00 | Quota Reset To: 9999 (+9329 jump from backup key transition)
Timestamp: 2026-10-03T20:01:18.569218+00:00 | Quota Reset To: 7600 (+7096 jump from backup key transition)
